# Working with Viewports

Viewports are the spatial backbone of grid_py. Every drawing operation
takes place inside the *current viewport*, which defines the coordinate
system, clipping region, and default graphical parameters.

This notebook covers:

* Pushing and popping viewports.
* Navigating the viewport tree with `up_viewport`, `down_viewport`, and `seek_viewport`.
* Building compound viewport structures with `VpStack`, `VpList`, and `VpTree`.

The narrative follows the original R vignette (`viewports.Rnw`).

In [ ]:
import matplotlib
matplotlib.use("Agg")

import grid_py as gp

## Push and Pop

`push_viewport` makes a viewport the current one (all subsequent
drawing happens inside it). `pop_viewport` removes the current
viewport and returns to its parent.

In [ ]:
gp.grid_newpage()

# Push a viewport covering the left half of the page
vp_left = gp.Viewport(
    x=gp.Unit(0.25, "npc"),
    width=gp.Unit(0.5, "npc"),
    name="left_half",
)
gp.push_viewport(vp_left)
gp.grid_rect(gp=gp.Gpar(fill="lightyellow"))
gp.grid_text("left", gp=gp.Gpar(fontsize=14))
gp.pop_viewport()

# Push a viewport covering the right half
vp_right = gp.Viewport(
    x=gp.Unit(0.75, "npc"),
    width=gp.Unit(0.5, "npc"),
    name="right_half",
)
gp.push_viewport(vp_right)
gp.grid_rect(gp=gp.Gpar(fill="lavender"))
gp.grid_text("right", gp=gp.Gpar(fontsize=14))
gp.pop_viewport()

## upViewport and downViewport

`up_viewport(n)` moves up *n* levels in the viewport tree without
destroying any viewports. `down_viewport(name)` descends to a named
child viewport. Together they let you revisit previously pushed
viewports.

In [ ]:
gp.grid_newpage()

# Build a small tree of viewports
outer = gp.Viewport(width=gp.Unit(0.9, "npc"), height=gp.Unit(0.9, "npc"), name="outer")
gp.push_viewport(outer)
gp.grid_rect(gp=gp.Gpar(col="grey", lty="dashed"))

inner = gp.Viewport(
    width=gp.Unit(0.5, "npc"), height=gp.Unit(0.5, "npc"),
    name="inner",
)
gp.push_viewport(inner)
gp.grid_rect(gp=gp.Gpar(fill="mistyrose"))
gp.grid_text("inner", gp=gp.Gpar(fontsize=12))

# Go back up one level
gp.up_viewport(1)

# Now descend back into 'inner'
gp.down_viewport("inner")
gp.grid_text("revisited", y=gp.Unit(0.3, "npc"), gp=gp.Gpar(fontsize=10, col="red"))

# Return to root
gp.up_viewport(0)  # 0 means go all the way up

## seekViewport

`seek_viewport(name)` searches the entire viewport tree for a viewport
with the given name and makes it the current viewport, regardless of
where you are in the tree.

In [ ]:
gp.grid_newpage()

# Create and push several nested viewports
gp.push_viewport(gp.Viewport(name="A", width=gp.Unit(0.8, "npc"), height=gp.Unit(0.8, "npc")))
gp.grid_rect(gp=gp.Gpar(col="grey"))

gp.push_viewport(gp.Viewport(name="B", width=gp.Unit(0.6, "npc"), height=gp.Unit(0.6, "npc")))
gp.grid_rect(gp=gp.Gpar(col="steelblue"))

gp.push_viewport(gp.Viewport(name="C", width=gp.Unit(0.4, "npc"), height=gp.Unit(0.4, "npc")))
gp.grid_rect(gp=gp.Gpar(fill="lightgreen"))
gp.grid_text("C")

# Jump directly to viewport A from inside C
gp.seek_viewport("A")
gp.grid_text("back in A", y=gp.Unit(0.1, "npc"), gp=gp.Gpar(fontsize=10, col="red"))

gp.up_viewport(0)

## VpStack, VpList, and VpTree

These container classes let you describe viewport structures declaratively
before pushing them:

* **VpStack** -- viewports pushed sequentially (each inside the previous one).
* **VpList** -- viewports pushed as siblings (in parallel).
* **VpTree** -- a parent viewport with a VpList of children.

In [ ]:
# VpStack: nested viewports
stack = gp.VpStack(
    gp.Viewport(name="s_outer", width=gp.Unit(0.9, "npc"), height=gp.Unit(0.9, "npc")),
    gp.Viewport(name="s_inner", width=gp.Unit(0.5, "npc"), height=gp.Unit(0.5, "npc")),
)
print("VpStack:", stack)

In [ ]:
# VpList: sibling viewports
vp_list = gp.VpList(
    gp.Viewport(name="panel_1", x=gp.Unit(0.25, "npc"), width=gp.Unit(0.4, "npc")),
    gp.Viewport(name="panel_2", x=gp.Unit(0.75, "npc"), width=gp.Unit(0.4, "npc")),
)
print("VpList:", vp_list)

In [ ]:
# VpTree: parent with children
tree = gp.VpTree(
    parent=gp.Viewport(name="parent", width=gp.Unit(0.9, "npc"), height=gp.Unit(0.9, "npc")),
    children=gp.VpList(
        gp.Viewport(name="child_a", x=gp.Unit(0.25, "npc"), width=gp.Unit(0.4, "npc")),
        gp.Viewport(name="child_b", x=gp.Unit(0.75, "npc"), width=gp.Unit(0.4, "npc")),
    ),
)
print("VpTree:", tree)

In [ ]:
# Push the VpTree and draw inside each child
gp.grid_newpage()
gp.push_viewport(tree)

# After pushing a VpTree, we are in the last child (child_b)
gp.grid_rect(gp=gp.Gpar(fill="lavender"))
gp.grid_text("child_b")

# Navigate to child_a
gp.seek_viewport("child_a")
gp.grid_rect(gp=gp.Gpar(fill="honeydew"))
gp.grid_text("child_a")

gp.up_viewport(0)

## Summary

* `push_viewport` / `pop_viewport` create and destroy viewport contexts.
* `up_viewport` / `down_viewport` navigate without destroying viewports.
* `seek_viewport` jumps to any named viewport in the tree.
* `VpStack`, `VpList`, and `VpTree` describe viewport structures declaratively.